# Day 8 · Lab 1 — End-to-End Multi-Agent BA Workflow

## What you'll build

1. Define BASessionState schema for the full workflow
2. Build 3 specialist agents (simplified): Data Query, Requirements, Report
3. Build a Supervisor that routes between them
4. Run against a sample Q4 review request
5. Trace the state through all agents

## Prerequisites

- Days 5-7 completed
- Same sandbox setup
- No new packages

## Step 1 — Environment

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

for pkg in ["python-dotenv", "langchain-openai", "langgraph"]:
    try: __import__(pkg.replace("-", "_").split("[")[0])
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pkg])

from dotenv import load_dotenv
load_dotenv(Path("~/agentic-lab/.env").expanduser(), override=False)
for k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY"):
    if os.environ.get(k) == "": del os.environ[k]

os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
print("✓ Ready")

## Step 2 — Shared state schema

In [ ]:
from typing import TypedDict, Annotated
import operator


class BASessionState(TypedDict):
    request: str
    sql_queries: Annotated[list, operator.add]
    sql_results: Annotated[list, operator.add]
    requirements: list
    insights: list
    final_report: str
    current_agent: str
    completed: Annotated[list, operator.add]


print("✓ BASessionState defined")
print(f"  Fields: {list(BASessionState.__annotations__.keys())}")

## Step 3 — Specialist agents (simplified for demo)

In real capstone, each specialist has the full Day 5/6/7 pipeline. Here we simulate briefly.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)


def data_query_agent(state: BASessionState) -> dict:
    print("  → data_query_agent running")
    # Simulated: 2 SQL queries returning canned data
    return {
        "sql_queries": [
            "SELECT region, SUM(revenue) FROM orders WHERE quarter='Q4' GROUP BY region",
            "SELECT region, AVG(csat) FROM customer_scores WHERE quarter='Q4' GROUP BY region",
        ],
        "sql_results": [
            {"NA": 8_200_000, "EU": 3_100_000, "APAC": 1_100_000},
            {"NA": 8.4, "EU": 7.9, "APAC": 8.9},
        ],
        "completed": ["data_query"],
    }


def requirements_agent(state: BASessionState) -> dict:
    print("  → requirements_agent running")
    # Simulated: recall relevant requirements from Day 6 store
    return {
        "requirements": [
            {"id": "REQ-01", "type": "business", "statement": "APAC 40% YoY revenue growth target"},
            {"id": "REQ-02", "type": "business", "statement": "Reduce EU churn below 3.0%"},
        ],
        "completed": ["requirements"],
    }


def report_agent(state: BASessionState) -> dict:
    print("  → report_agent running")
    prompt = f'''Write a 2-paragraph Q4 executive summary based on:

Data: {state['sql_results']}
Stated goals: {[r['statement'] for r in state['requirements']]}

Focus on: what would surprise a CFO. Preserve numbers. No adjectives without data.'''
    report = llm.invoke(prompt).content.strip()
    return {"final_report": report, "completed": ["report"]}


print("✓ 3 specialist agents defined")

## Step 4 — Supervisor routing

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


def supervisor_router(state: BASessionState) -> str:
    done = state.get("completed", [])
    if "data_query" not in done:
        return "data_query"
    if "requirements" not in done:
        return "requirements"
    if "report" not in done:
        return "report"
    return END


builder = StateGraph(BASessionState)
builder.add_node("data_query", data_query_agent)
builder.add_node("requirements", requirements_agent)
builder.add_node("report", report_agent)

builder.add_conditional_edges(
    START,
    supervisor_router,
    {"data_query": "data_query", "requirements": "requirements", "report": "report", END: END},
)
builder.add_conditional_edges(
    "data_query",
    supervisor_router,
    {"data_query": "data_query", "requirements": "requirements", "report": "report", END: END},
)
builder.add_conditional_edges(
    "requirements",
    supervisor_router,
    {"data_query": "data_query", "requirements": "requirements", "report": "report", END: END},
)
builder.add_conditional_edges(
    "report",
    supervisor_router,
    {"data_query": "data_query", "requirements": "requirements", "report": "report", END: END},
)

graph = builder.compile(checkpointer=MemorySaver())
print("✓ Multi-agent graph compiled")

## Step 5 — Run end-to-end

In [ ]:
import uuid

thread_id = f"ba-{uuid.uuid4().hex[:8]}"
config = {"configurable": {"thread_id": thread_id}}

initial = {
    "request": "Give me a Q4 executive review",
    "sql_queries": [],
    "sql_results": [],
    "requirements": [],
    "insights": [],
    "final_report": "",
    "current_agent": "",
    "completed": [],
}

print("Running end-to-end workflow...")
print()
final = graph.invoke(initial, config)
print()
print("═" * 60)
print("WORKFLOW COMPLETE")
print("═" * 60)
print(f"Completed agents: {final['completed']}")
print(f"SQL queries run: {len(final['sql_queries'])}")
print(f"Requirements pulled: {len(final['requirements'])}")
print(f"Final report length: {len(final['final_report'])} chars")
print()
print("─── FINAL REPORT ───")
print(final['final_report'])

## Step 6 — Inspect the state history

In [ ]:
history = list(graph.get_state_history(config))
print(f"State transitions: {len(history)}")
print()
for i, snap in enumerate(reversed(history)):
    next_node = snap.next[0] if snap.next else "END"
    completed = snap.values.get("completed", [])
    print(f"Step {i}: next={next_node}, completed={completed}")

## What you learned

1. **Supervisor pattern**: routing function decides which specialist next based on completed list
2. **BASessionState**: shared typed state that all agents read/write
3. **`Annotated[list, operator.add]`**: lets multiple agents append to same field without overwriting
4. **Conditional edges**: implement supervisor routing after every specialist runs

## Production notes

- Add HITL (`interrupt_before`) for high-stakes reports
- Add error handling in each specialist (retry, fallback, escalate)
- Add PostgresSaver for durable state
- Add LangSmith @traceable or OTel spans for observability

Next: open Lab 2 to design your capstone.